In [1]:
# 필요하면 SASRec distillation 추가
# 이건 maxsim 코든데... 원리도 모르고 뭐 암껏도 모르거든?
# 그리고 제미나이형님이 내 데이터 같은 경우엔 maxsim 굳이 하지 말랬는데....
# 일단 한번 돌려보고 별 차이 없으면 쳐내자 얘는
import random
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data import random_split

from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType #, AdapterConfig

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)            # 기본 Python random 고정
    np.random.seed(seed)         # NumPy 랜덤 고정
    torch.manual_seed(seed)      # CPU 연산 랜덤 고정
    torch.cuda.manual_seed(seed) # GPU 모든 디바이스 랜덤 고정
    torch.cuda.manual_seed_all(seed)  # 멀티 GPU일 때

    # 연산 재현성
    torch.backends.cudnn.deterministic = True  # cuDNN 연산을 determinisitc으로 강제
    torch.backends.cudnn.benchmark = False     # CUDA 성능 자동 튜닝 기능 끔 → 완전 재현 가능
    
set_seed(42)

In [3]:
# model_name = "intfloat/e5-small" 
model_name = "intfloat/e5-small" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, lora_config)

In [4]:
# book_path = './data/e5_book_meta.parquet'
book_path = './data/book_for_emb.parquet'
books = pd.read_parquet(book_path)

In [5]:
# 100개 미만인 카테고리는 노이즈로 간주하고 삭제 추천
counts = books['category'].value_counts()
valid_categories = counts[counts > 100].index
books = books[books['category'].isin(valid_categories)]

In [6]:
# 입력 텍스트 생성 (타이틀 + 설명 + 저자 등 결합)
def build_text(row):
    parts = [
        f"Title: {row['title']} |",
        # f"Category: {row['category']} |",
        f"Description: {row['description']}"
    ]
    return " ".join( # 리스트의 문자열들을 공백으로 연결할건데.....
        [p for p in parts if isinstance(p, str)] # NaN이나 None이 있으면 제외함
    ) # 최종적으로 하나의 문장 형태로 반환한다고 함!! "Title: ... Category: ... Description: ..."

books["text"] = books.apply(build_text, axis=1) # 새 컬럼 text에 대해서.... 문장 만듦

In [7]:
def chunk_text(text, tokenizer, max_length=512, stride=64):
    """
    텍스트를 max_length 길이로 자르되, stride만큼 겹치게(overlap) 이동하며 자름.
    반환값: 텍스트 조각(chunk)들의 리스트
    """
    tokens = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
    
    # 텍스트가 max_length보다 짧으면 그냥 리턴
    if len(tokens) <= max_length:
        return [text]
    
    chunk_texts = []
    # Sliding Window
    for i in range(0, len(tokens), max_length - stride):
        chunk_ids = tokens[i : i + max_length]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        chunk_texts.append(chunk_text)
        
        # 끝까지 다다르면 종료
        if i + max_length >= len(tokens):
            break
            
    return chunk_texts

print("Chunking text...")
# 'chunks' 컬럼에 리스트 형태로 저장됨
tqdm.pandas()
books['chunks'] = books['text'].progress_apply(lambda x: chunk_text(x, tokenizer, max_length=512, stride=64))

# 원본 책 ID 보존 (나중에 MaxSim 평가할 때, 어떤 Chunk들이 같은 책인지 알기 위함)
books['book_id'] = books.index 

# --- [수정 3] Explode: 리스트를 여러 행으로 펼치기 ---
# 이제 행의 개수가 확 늘어납니다.
books_exploded = books.explode('chunks').reset_index(drop=True)
books_exploded = books_exploded.rename(columns={'chunks': 'chunk_text'})

print(f"Original Rows: {len(books)} -> Exploded Rows: {len(books_exploded)}")

# Label Encoding (데이터가 늘어났어도 카테고리는 동일)
le = LabelEncoder()
books_exploded['label_idx'] = le.fit_transform(books_exploded['category'])
num_classes = len(le.classes_)

# Dataset 생성 (exploded 된 데이터프레임 사용)
dataset = Dataset.from_pandas(books_exploded[['book_id', 'chunk_text', 'label_idx']])

Chunking text...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 81845/81845 [00:24<00:00, 3357.52it/s]


Original Rows: 81845 -> Exploded Rows: 83759


In [8]:
def collate_fn(batch):
    texts = [f"passage: {x['chunk_text']}" for x in batch]
    labels = torch.tensor([x['label_idx'] for x in batch])

    inputs = tokenizer(
      texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
    
    return inputs, labels

In [9]:
total_len = len(dataset)
train_len = int(total_len * 0.8)
valid_len = total_len - train_len

train_dataset, valid_dataset = random_split(dataset, [train_len, valid_len])

train_loader = DataLoader(
    train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn
)

In [10]:
def compute_retrieval_accuracy(model, dataloader, device, k=10):
    embeddings_list = []
    labels_list = []

    with torch.no_grad():
        for batch_inputs, labels in dataloader:
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            labels = labels.to(device)

            outputs = model(**batch_inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            embeddings = F.normalize(embeddings, p=2, dim=1)

            embeddings_list.append(embeddings.cpu()) # CPU로 옮겨서 메모리 절약
            labels_list.append(labels.cpu())

    all_embeddings = torch.cat(embeddings_list, dim=0)
    all_labels = torch.cat(labels_list, dim=0)
    
    # 데이터가 많으면 matmul에서 OOM 날 수 있으니 배치 처리가 좋으나, 여기선 간단히 진행
    # (주의: Explode로 인해 데이터가 3~4배 늘었으므로 메모리 주의)
    
    # Chunk 단위 정확도 계산
    similarity = torch.matmul(all_embeddings, all_embeddings.T)
    similarity.fill_diagonal_(-1e9)

    topk_vals, topk_idx = similarity.topk(k, dim=1)
    nn_labels_topk = all_labels[topk_idx] 

    # -----------------------------
    # top-1: 가장 가까운 neighbor 정답 여부
    # top-k: k개 안에 정답이 하나라도 있으면
    # precision@k: k개 neighbor 중 정답 비율 평균
    # MRR: 정답이 rank 몇 번째인지에 따른 평균 역수 → rank 1이면 1.0, rank 2이면 0.5
    # -----------------------------
    # Top-1 accuracy
    top1_labels = nn_labels_topk[:, 0]
    top1_acc = (top1_labels == all_labels).float().mean().item()
    
    # Top-k accuracy (at least 1 match)
    topk_match = (nn_labels_topk == all_labels.unsqueeze(1)).float().mean(dim=1) # .any(dim=1)
    topk_acc = topk_match.float().mean().item()

    # Precision@k
    precision_at_k = (nn_labels_topk == all_labels.unsqueeze(1)).float().mean(dim=1).mean().item()
    
    # MRR (Mean Reciprocal Rank)
    ranks = (nn_labels_topk == all_labels.unsqueeze(1)).float() # 정답 label 위치 찾기
    reciprocal_rank = []
    for i in range(ranks.size(0)):
        pos_positions = torch.nonzero(ranks[i]).flatten()
        if len(pos_positions) == 0: # positive 없으면 reciprocal rank = 0
            reciprocal_rank.append(0.0)
        else:
            reciprocal_rank.append(1.0 / (pos_positions[0].item() + 1))
    mrr = sum(reciprocal_rank) / len(reciprocal_rank)

    metrics = {
        "top1_acc": top1_acc,
        "topk_acc": topk_acc,
        "precision@k": precision_at_k,
        "mrr": mrr
    }
    return metrics

In [11]:
def hard_negative(embeddings, labels, k=3):
    batch_size = embeddings.size(0)
    similarity = torch.matmul(embeddings, embeddings.T) # 1. 유사도 행렬 계산
    pos_mask = labels.unsqueeze(1) == labels.unsqueeze(0)
    similarity.masked_fill_(pos_mask, -1e9) 
    hard_neg_sims, _ = similarity.topk(k, dim=1) # (N, k)

    return hard_neg_sims

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

EPOCHS = 20
WARMUP_RATIO = 0.1 # 전체 스텝의 10%를 웜업에 사용
LEARNING_RATE = 3e-5 # 3e-4는 maxsim 없을땐데.... maxsim 넣으면 3e-5가 더 나은가봄 

total_steps = len(train_loader) * EPOCHS

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps
)

best_mrr = 0.0
save_path = "./lora_best_retrieval"

In [13]:
import wandb

wandb.init(
    project="retrieval-contrastive", 
    name="e5-lora-maxsim-v2",
    config={
        "lr": 3e-4,
        "temperature": 0.05,
        "epochs": 20,
        "batch_size": train_loader.batch_size,
    }
)

wandb.watch(model, log="all")  # gradients, parameters 기록

/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using

In [ ]:
metrics = compute_retrieval_accuracy(model, valid_loader, device)
wandb.log({
    "epoch": 0,
    "train_loss": 2,
    "valid/top1_acc": metrics["top1_acc"],
    "valid/top10_acc": metrics["topk_acc"],
    "valid/precision@10": metrics["precision@k"],
    "valid/mrr": metrics["mrr"],
    "lr": 3e-4,
})
print(f"[Before Training...]")
print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR             : {metrics['mrr']:.4f}")

for epoch in range(20):
    model.train()
    total_train_loss = 0

    for batch_inputs, labels in tqdm(train_loader, desc = f"Epoch: {epoch+1}"):
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        labels = labels.to(device)
        
        outputs = model(**batch_inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        
        similarity = torch.matmul(embeddings, embeddings.T)
        
        labels_eq = labels.unsqueeze(1) == labels.unsqueeze(0)
        identity_mask = torch.eye(len(labels), device=labels.device).bool() # 자기 자신 제거 mask
        labels_eq = labels_eq & (~identity_mask)

        pos_mask = labels_eq.float()   # (N, N)

        # Positive가 하나도 없는 배치가 있을 수 있음 (이 경우 Loss 0 처리 혹은 회피)
        if pos_mask.sum() == 0:
            continue
        
        pos_sim = similarity * pos_mask # (N, N) → anchor i의 positive 여러 개 (동일 label 여러 개)
        neg_sim = hard_negative(embeddings, labels) # (N, k) → anchor i의 negative k개

        temperature = 0.05 # 타우
        # InfoNCE / Contrastive Loss LogSumExp Trick 적용 권장하나 기존 코드 유지
        # 분모: Positive들의 합 + Negative들의 합
        exp_pos = torch.exp(pos_sim / temperature) * pos_mask # 마스킹된 부분만 exp
        exp_neg = torch.exp(neg_sim / temperature)
        
        # 각 anchor에 대해 positive가 있는 경우만 loss 계산
        denominator = exp_pos.sum(dim=1) + exp_neg.sum(dim=1)
        numerator = exp_pos.sum(dim=1)
        
        # 0으로 나누기 방지
        mask_valid = (numerator > 0)
        loss = -torch.log(numerator[mask_valid] / denominator[mask_valid] + 1e-8).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_train_loss += loss.item()
        
    train_loss = total_train_loss / len(train_loader)
  
    model.eval()
    metrics = compute_retrieval_accuracy(model, valid_loader, device)
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "valid/top1_acc": metrics["top1_acc"],
        "valid/top10_acc": metrics["topk_acc"],
        "valid/precision@10": metrics["precision@k"],
        "valid/mrr": metrics["mrr"],
        "lr": scheduler.get_last_lr()[0],
    })
    print(f"[Epoch {epoch + 1}] Train Loss: {train_loss:.4f}")
    print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
    print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR             : {metrics['mrr']:.4f}")

[Before Training...]
Top-1 Accuracy : 0.4685 | Top-10 Accuracy : 0.3562
Precision@10   : 0.3562 | MRR             : 0.5924


Epoch: 1: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 524/524 [01:49<00:00,  4.80it/s]


[Epoch 1] Train Loss: 0.8513
Top-1 Accuracy : 0.4817 | Top-10 Accuracy : 0.3697
Precision@10   : 0.3697 | MRR             : 0.6053


Epoch: 2: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 524/524 [01:49<00:00,  4.77it/s]


[Epoch 2] Train Loss: 0.5962
Top-1 Accuracy : 0.4277 | Top-10 Accuracy : 0.3326
Precision@10   : 0.3326 | MRR             : 0.5580


Epoch: 3: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 524/524 [01:49<00:00,  4.79it/s]
